# Chapter 11: Aggregate Loss Models

Code of Chapter 11 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

The walkthrough comes first, because it loads the data and imports used by the other examples of the chapter.

## 11.7 Python Walkthrough

In [ ]:
import numpy as np
from scipy import stats
from gidata import synthetic_portfolio
from gilossmodels import fit_mle
from giaggregate import (compound_moments, compound_poisson_skewness, lognormal_raw_moment,
                         discretize, panjer, fft_aggregate, simulate, grid_quantile,
                         grid_tvar, grid_moments, normal_approx, translated_gamma,
                         lognormal_approx, sample_tvar)

# Section 11.2.3: the hand example
panjer("poisson", np.array([0, 0.5, 0.3, 0.2]), 4, lam=2)   # 0.135335 0.135335 0.148869 0.157891

# Section 11.1.3: severity and moments for the book of 500 policies
_, claims = synthetic_portfolio()
L = fit_mle(claims["ultimate_amount"].to_numpy(), "lognormal")
m1, m2, m3 = (lognormal_raw_moment(L["mu"], L["sigma"], k) for k in (1, 2, 3))
ES, VS = compound_moments(48.5, 48.5, m1, m2 - m1**2)        # 195917, 3.1618e9
sk = compound_poisson_skewness(48.5, m2, m3)                 # 1.147

# Section 11.2.4 and 11.3: three methods
X = stats.lognorm(L["sigma"], scale=np.exp(L["mu"]))         # scipy: s = sigma, scale = e^mu
fx = discretize(X.cdf, 200, 2**13)
g = panjer("poisson", fx, 2**13, lam=48.5)
gf = fft_aggregate("poisson", fx, 2**14, lam=48.5)[:2**13]
[grid_quantile(g, 200, q) for q in (0.5, 0.9, 0.95, 0.99, 0.995)]
grid_tvar(g, 200, 0.99)                                      # 415682
sims = simulate(100_000, lambda r, k: r.poisson(48.5, k),
                lambda r, k: r.lognormal(L["mu"], L["sigma"], k), seed=11)

# Section 11.4: approximations
TG, (alpha, beta, x0) = translated_gamma(ES, VS, sk)
[round(TG.ppf(q)) for q in (0.5, 0.9, 0.95, 0.99, 0.995)]

## Exercise 2: Overdispersion in the book of 500 policies

In [ ]:
nb = dict(r=10, p=10 / 58.5)
g = panjer("negbin", fx, 2**13, **nb)
[grid_quantile(g, 200, q) for q in (0.90, 0.99, 0.995)]      # 306400, 441800, 480600